# NetSentinel - Detector 3: C2 Beaconing (BiLSTM + FFT)

**Model**: Bidirectional LSTM with FFT Periodicity Features 
**Dataset**: CTU-13 (botnet scenarios) + CIC-IDS2017 (benign traffic) 
**Task**: Binary (beacon vs non-beacon) 
**Key Innovation**: FFT extracts hidden periodic signals in connection timing - catches both precise and jittered beacons 
**Export**: ONNX

---

In [ ]:
# Only install what Kaggle doesn't already have
!pip install -q onnxruntime

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os, json, time, random, glob, warnings
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

OUTPUT_DIR = '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEQ_LEN = 100
N_FEATURES = 4
MIN_FLOWS = SEQ_LEN

## 1. Data Loading - CTU-13 + CIC-IDS2017

In [ ]:
# ============================================================
# Helper: find column by keyword matching
# ============================================================

def find_col(columns, keywords):
    """Find a column name containing any keyword (case/space/underscore insensitive)."""
    for c in columns:
        cl = c.strip().lower().replace(' ', '').replace('_', '')
        for kw in keywords:
            if kw in cl:
                return c
    return None

In [ ]:
# ============================================================
# Scan dataset directories
# ============================================================

CTU13_DIR = '/kaggle/input/datasets/dhoogla/ctu13'
CICIDS_DIR = '/kaggle/input/datasets/dhoogla/cicids2017'

# List everything under both dirs to understand structure
print('=== CTU-13 directory ===')
for root, dirs, files in os.walk(CTU13_DIR):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp) / 1024 / 1024
        print(f'  {fp}  ({sz:.1f} MB)')

print('\n=== CIC-IDS2017 directory ===')
for root, dirs, files in os.walk(CICIDS_DIR):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp) / 1024 / 1024
        print(f'  {fp}  ({sz:.1f} MB)')

In [ ]:
# ============================================================
# Load CTU-13
# ============================================================

ctu_files = []
for root, dirs, files in os.walk(CTU13_DIR):
    for f in files:
        if f.endswith(('.csv', '.parquet', '.binetflow')):
            ctu_files.append(os.path.join(root, f))

print(f'CTU-13 data files: {len(ctu_files)}')

ctu_dfs = []
for f in ctu_files:
    try:
        if f.endswith('.parquet'):
            df = pd.read_parquet(f)
        else:
            df = pd.read_csv(f, low_memory=False)
        df.columns = df.columns.str.strip()
        label_col = find_col(df.columns, ['label', 'class', 'tag', 'category'])
        if label_col:
            ctu_dfs.append(df)
            print(f'  OK: {os.path.basename(f)} -> {len(df):,} rows')
            print(f'      Columns: {list(df.columns)}')
            print(f'      Labels:  {df[label_col].value_counts().head(5).to_dict()}')
        else:
            print(f'  SKIP: {os.path.basename(f)} (no label col). Cols: {list(df.columns)}')
    except Exception as e:
        print(f'  ERR: {os.path.basename(f)}: {str(e)[:100]}')

if ctu_dfs:
    ctu_all = pd.concat(ctu_dfs, ignore_index=True)
    ctu_all.columns = ctu_all.columns.str.strip()
    print(f'\nTotal CTU-13: {len(ctu_all):,} flows')
else:
    print('WARNING: No CTU-13 data loaded!')
    ctu_all = pd.DataFrame()

In [ ]:
# ============================================================
# Load CIC-IDS2017 (benign only)
# ============================================================

cic_files = []
for root, dirs, files in os.walk(CICIDS_DIR):
    for f in files:
        if f.endswith(('.csv', '.parquet')):
            cic_files.append(os.path.join(root, f))

print(f'CIC-IDS2017 files: {len(cic_files)}')

cic_dfs = []
for f in cic_files:
    try:
        if f.endswith('.parquet'):
            df = pd.read_parquet(f)
        else:
            df = pd.read_csv(f, low_memory=False)
        df.columns = df.columns.str.strip()
        label_col = find_col(df.columns, ['label', 'class', 'tag', 'category'])
        if label_col:
            benign_mask = df[label_col].astype(str).str.lower().str.contains('benign')
            df_b = df[benign_mask].copy()
            if len(df_b) > 0:
                cic_dfs.append(df_b)
                print(f'  OK: {os.path.basename(f)} -> {len(df_b):,} benign')
        else:
            print(f'  SKIP: {os.path.basename(f)}')
    except Exception as e:
        print(f'  ERR: {os.path.basename(f)}: {str(e)[:80]}')

if cic_dfs:
    cic_benign = pd.concat(cic_dfs, ignore_index=True)
    cic_benign.columns = cic_benign.columns.str.strip()
    print(f'\nTotal CIC-IDS2017 benign: {len(cic_benign):,}')
    print(f'Columns sample: {list(cic_benign.columns[:8])}...')
else:
    print('WARNING: No CIC-IDS2017 data!')
    cic_benign = pd.DataFrame()

## 2. Extract Flow Sequences

In [ ]:
# ============================================================
# Extract sequences from CTU-13
# ============================================================

def extract_sequences_ctu(df, label_value, max_sequences=15000):
    """Extract flow sequences from CTU-13 with auto column detection."""
    sequences = []
    if len(df) == 0:
        return sequences
    
    # Auto-detect columns
    label_col = find_col(df.columns, ['label', 'class', 'tag'])
    time_col  = find_col(df.columns, ['starttime', 'timestamp', 'time'])
    pkts_col  = find_col(df.columns, ['totpkt', 'totalpkt', 'totalpacket'])
    bytes_col = find_col(df.columns, ['totbyte', 'totalbyte'])
    sbytes_col = find_col(df.columns, ['srcbyte', 'sourcebyte'])
    dir_col   = find_col(df.columns, ['dir'])
    src_col   = find_col(df.columns, ['srcaddr', 'srcip', 'sourceip', 'sourceaddr'])
    dst_col   = find_col(df.columns, ['dstaddr', 'dstip', 'destip', 'destaddr', 'destinationip'])
    dur_col   = find_col(df.columns, ['dur', 'duration', 'flowduration'])
    
    print(f'  Columns found: label={label_col} time={time_col} pkts={pkts_col} bytes={bytes_col} src={src_col} dst={dst_col} dur={dur_col}')
    
    if not label_col:
        print(f'  ERROR: no label column. All columns: {list(df.columns)}')
        return sequences
    
    # Filter by label
    labels_str = df[label_col].astype(str).str.lower()
    if label_value == 1:
        mask = labels_str.str.contains('botnet|malicious|bot|c2|attack|infected', na=False)
    else:
        mask = labels_str.str.contains('normal|benign|legit|background', na=False)
    
    filtered = df[mask].copy()
    print(f'  Matched rows: {len(filtered):,}')
    
    if len(filtered) == 0:
        print(f'  Unique labels: {df[label_col].unique()[:20]}')
        return sequences
    
    # Parse timestamps
    if time_col and time_col in filtered.columns:
        filtered['_ts'] = pd.to_datetime(filtered[time_col], errors='coerce', dayfirst=True)
        # Try unix timestamp if datetime parsing failed for most rows
        if filtered['_ts'].isna().sum() > len(filtered) * 0.5:
            filtered['_ts'] = pd.to_datetime(pd.to_numeric(filtered[time_col], errors='coerce'), unit='s', errors='coerce')
    elif dur_col and dur_col in filtered.columns:
        dur_vals = pd.to_numeric(filtered[dur_col], errors='coerce').fillna(0).cumsum()
        filtered['_ts'] = pd.Timestamp('2020-01-01') + pd.to_timedelta(dur_vals, unit='s')
    else:
        filtered['_ts'] = pd.date_range('2020-01-01', periods=len(filtered), freq='s')
    
    filtered = filtered.dropna(subset=['_ts']).sort_values('_ts')
    print(f'  After timestamp parse: {len(filtered):,} rows')
    
    # Parse numeric columns
    actual_bytes_col = bytes_col or sbytes_col
    for c in [pkts_col, actual_bytes_col]:
        if c and c in filtered.columns:
            filtered[c] = pd.to_numeric(filtered[c], errors='coerce').fillna(1)
    
    # Direction
    if dir_col and dir_col in filtered.columns:
        filtered['_dir'] = filtered[dir_col].astype(str).str.strip().isin(['->', '-->', 'forward', '1']).astype(float)
    else:
        filtered['_dir'] = 1.0
    
    # Group by conversation
    if src_col and dst_col and src_col in filtered.columns and dst_col in filtered.columns:
        groups = list(filtered.groupby([src_col, dst_col]))
    else:
        groups = [('all', filtered)]
    
    print(f'  Conversation pairs: {len(groups)}')
    
    for group_key, group_df in groups:
        if isinstance(group_key, tuple):
            grp = group_df
        else:
            grp = group_df
        
        if len(grp) < MIN_FLOWS:
            continue
        
        grp = grp.sort_values('_ts')
        ts = grp['_ts'].values.astype(np.int64) / 1e9
        iats = np.diff(ts)
        iats = np.clip(iats, 0.001, 86400)
        
        if pkts_col and pkts_col in grp.columns:
            pkt_vals = grp[pkts_col].values[1:]
        else:
            pkt_vals = np.random.lognormal(4, 1, len(iats))
        
        if actual_bytes_col and actual_bytes_col in grp.columns:
            byte_vals = grp[actual_bytes_col].values[1:]
        else:
            byte_vals = pkt_vals * np.random.uniform(50, 200, len(iats))
        
        dir_vals = grp['_dir'].values[1:]
        
        # Sliding window
        n = len(iats)
        step = SEQ_LEN // 2
        for start in range(0, n - SEQ_LEN + 1, step):
            end = start + SEQ_LEN
            seq = np.column_stack([
                iats[start:end], pkt_vals[start:end],
                byte_vals[start:end], dir_vals[start:end]
            ]).astype(np.float32)
            seq = np.nan_to_num(seq, nan=0.0, posinf=86400.0, neginf=0.0)
            sequences.append(seq)
            if len(sequences) >= max_sequences:
                return sequences
    
    return sequences

print('--- Extracting BOTNET sequences ---')
beacon_seqs = extract_sequences_ctu(ctu_all, label_value=1, max_sequences=15000)
print(f'Result: {len(beacon_seqs)} beacon sequences\n')

print('--- Extracting NORMAL sequences ---')
normal_seqs_ctu = extract_sequences_ctu(ctu_all, label_value=0, max_sequences=5000)
print(f'Result: {len(normal_seqs_ctu)} normal sequences')

In [ ]:
# ============================================================
# Extract sequences from CIC-IDS2017 (benign)
# ============================================================

def extract_sequences_cicids(df, max_sequences=10000):
    """Extract benign sequences from CIC-IDS2017."""
    sequences = []
    if len(df) == 0:
        return sequences
    
    iat_col   = find_col(df.columns, ['flowiatmean', 'iatmean'])
    pkt_col   = find_col(df.columns, ['fwdpacketlengthmean', 'avgpacketsize', 'packetlengthmean'])
    fwdp_col  = find_col(df.columns, ['totalfwdpacket', 'totalfwdpackets'])
    bwdp_col  = find_col(df.columns, ['totalbackwardpacket', 'totalbackwardpackets', 'totalbwdpackets'])
    fwdb_col  = find_col(df.columns, ['totallengthofffwd', 'fwdpacketslengthtotal', 'totallengthoffwdpacket'])
    dur_col   = find_col(df.columns, ['flowduration', 'duration'])
    
    print(f'  CIC cols: iat={iat_col}, pkt={pkt_col}, fwdp={fwdp_col}, dur={dur_col}')
    
    for c in [iat_col, pkt_col, fwdp_col, bwdp_col, fwdb_col, dur_col]:
        if c and c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan)
    
    df_s = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    
    for start in range(0, len(df_s) - SEQ_LEN, SEQ_LEN):
        chunk = df_s.iloc[start:start + SEQ_LEN]
        
        if iat_col and iat_col in chunk.columns:
            iats = np.clip(np.abs(chunk[iat_col].fillna(1).values / 1e6), 0.001, 86400)
        elif dur_col and fwdp_col and dur_col in chunk.columns:
            d = chunk[dur_col].fillna(1).values / 1e6
            p = chunk[fwdp_col].fillna(1).values + 1
            iats = np.clip(np.abs(d / p), 0.001, 86400)
        else:
            iats = np.random.exponential(5.0, SEQ_LEN)
        
        if pkt_col and pkt_col in chunk.columns:
            pkt_sizes = np.abs(chunk[pkt_col].fillna(100).values)
        else:
            pkt_sizes = np.random.lognormal(6, 1.5, SEQ_LEN)
        
        if fwdb_col and fwdb_col in chunk.columns:
            byte_counts = np.abs(chunk[fwdb_col].fillna(100).values)
        else:
            byte_counts = pkt_sizes * np.random.uniform(0.5, 2.0, SEQ_LEN)
        
        if fwdp_col and bwdp_col and fwdp_col in chunk.columns and bwdp_col in chunk.columns:
            fwd = np.abs(chunk[fwdp_col].fillna(0).values) + 1
            bwd = np.abs(chunk[bwdp_col].fillna(0).values) + 1
            directions = (fwd > bwd).astype(float)
        else:
            directions = np.random.randint(0, 2, SEQ_LEN).astype(float)
        
        seq = np.column_stack([iats, pkt_sizes, byte_counts, directions]).astype(np.float32)
        seq = np.nan_to_num(seq, nan=0.0, posinf=86400.0, neginf=0.0)
        sequences.append(seq)
        if len(sequences) >= max_sequences:
            break
    
    return sequences

print('--- Extracting BENIGN sequences from CIC-IDS2017 ---')
benign_seqs_cic = extract_sequences_cicids(cic_benign, max_sequences=10000)
print(f'Result: {len(benign_seqs_cic)} benign sequences')

In [ ]:
# ============================================================
# Combine + synthetic augmentation
# ============================================================

def gen_beacon(n=SEQ_LEN):
    base = random.choice([30, 60, 120, 180, 300, 600, 900, 1800, 3600])
    jit = random.uniform(0, 0.20)
    iats = [max(1, base + np.random.normal(0, base * jit)) for _ in range(n)]
    sz = random.randint(50, 500)
    ps = [max(20, sz + np.random.normal(0, sz * 0.1)) for _ in range(n)]
    bs = [max(10, s * random.uniform(0.5, 1.5)) for s in ps]
    ds = [1 if random.random() > 0.3 else 0 for _ in range(n)]
    return np.column_stack([iats, ps, bs, ds]).astype(np.float32)

def gen_benign(n=SEQ_LEN):
    iats = [max(0.01, np.random.exponential(random.uniform(0.5, 30)) if random.random() > 0.3 else random.uniform(60, 7200)) for _ in range(n)]
    ps = [np.random.lognormal(6, 1.5) for _ in range(n)]
    bs = [np.random.lognormal(7, 2) for _ in range(n)]
    ds = [random.randint(0, 1) for _ in range(n)]
    return np.column_stack([iats, ps, bs, ds]).astype(np.float32)

TARGET = 15000

all_beacon = list(beacon_seqs)
need_b = max(0, TARGET - len(all_beacon))
if need_b > 0:
    print(f'Adding {need_b} synthetic beacon sequences...')
    all_beacon += [gen_beacon() for _ in tqdm(range(need_b))]

all_benign = list(benign_seqs_cic) + list(normal_seqs_ctu)
need_n = max(0, TARGET - len(all_benign))
if need_n > 0:
    print(f'Adding {need_n} synthetic benign sequences...')
    all_benign += [gen_benign() for _ in tqdm(range(need_n))]

n_cls = min(len(all_beacon), len(all_benign), TARGET)
random.shuffle(all_beacon)
random.shuffle(all_benign)
all_beacon = all_beacon[:n_cls]
all_benign = all_benign[:n_cls]

sequences = np.array(all_beacon + all_benign)
labels = np.array([1]*len(all_beacon) + [0]*len(all_benign))

print(f'\nDataset: {sequences.shape}')
print(f'Labels:  {Counter(labels)}')
print(f'Real beacon:  {len(beacon_seqs)}')
print(f'Real benign:  {len(benign_seqs_cic) + len(normal_seqs_ctu)}')
print(f'Synthetic:    {need_b + need_n}')

## 3. FFT Features + Normalization

In [ ]:
def extract_fft_features(iats):
    if len(iats) < 4:
        return np.zeros(5, dtype=np.float32)
    iats_norm = (iats - np.mean(iats)) / (np.std(iats) + 1e-8)
    fft_v = np.abs(np.fft.rfft(iats_norm))[1:]
    freqs = np.fft.rfftfreq(len(iats_norm))[1:]
    if len(fft_v) == 0:
        return np.zeros(5, dtype=np.float32)
    di = np.argmax(fft_v)
    te = np.sum(fft_v**2) + 1e-8
    psd = fft_v**2 / te + 1e-10
    hr = 0.0
    if di > 0:
        hi = [i for i in range(di, len(fft_v), di)]
        hr = float(sum(fft_v[i]**2 for i in hi if i < len(fft_v)) / te)
    return np.array([
        float(freqs[di]),
        float(fft_v[di]**2 / te),
        float(-np.sum(psd * np.log2(psd))),
        hr,
        float((fft_v[di] - np.mean(fft_v)) / (np.mean(fft_v) + 1e-8))
    ], dtype=np.float32)

print('Computing FFT features...')
fft_features = np.array([extract_fft_features(s[:, 0]) for s in tqdm(sequences)])
print(f'FFT shape: {fft_features.shape}')

bm = labels == 1
print(f'Periodicity -- Beacon: {fft_features[bm, 1].mean():.4f}, Benign: {fft_features[~bm, 1].mean():.4f}')
print(f'Entropy     -- Beacon: {fft_features[bm, 2].mean():.4f}, Benign: {fft_features[~bm, 2].mean():.4f}')

In [ ]:
# Normalize + split + loaders
scaler_seq = StandardScaler()
sequences_scaled = scaler_seq.fit_transform(sequences.reshape(-1, N_FEATURES)).reshape(sequences.shape)
scaler_fft = StandardScaler()
fft_scaled = scaler_fft.fit_transform(fft_features)

np.save(os.path.join(OUTPUT_DIR, 'scaler_seq_mean.npy'), scaler_seq.mean_)
np.save(os.path.join(OUTPUT_DIR, 'scaler_seq_scale.npy'), scaler_seq.scale_)
np.save(os.path.join(OUTPUT_DIR, 'scaler_fft_mean.npy'), scaler_fft.mean_)
np.save(os.path.join(OUTPUT_DIR, 'scaler_fft_scale.npy'), scaler_fft.scale_)

class BeaconDataset(Dataset):
    def __init__(self, seqs, ffts, labs):
        self.s = torch.tensor(seqs, dtype=torch.float32)
        self.f = torch.tensor(ffts, dtype=torch.float32)
        self.l = torch.tensor(labs, dtype=torch.long)
    def __len__(self): return len(self.l)
    def __getitem__(self, i): return self.s[i], self.f[i], self.l[i]

idx_tr, idx_te = train_test_split(np.arange(len(labels)), test_size=0.2, random_state=SEED, stratify=labels)
train_ds = BeaconDataset(sequences_scaled[idx_tr], fft_scaled[idx_tr], labels[idx_tr])
test_ds = BeaconDataset(sequences_scaled[idx_te], fft_scaled[idx_te], labels[idx_te])

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'Train: {len(train_ds):,} | Test: {len(test_ds):,}')

## 4. Model + Training

In [ ]:
class C2BeaconDetector(nn.Module):
    def __init__(self, seq_f=4, fft_f=5, h=64, nl=2, do=0.3):
        super().__init__()
        self.lstm = nn.LSTM(seq_f, h, nl, batch_first=True, bidirectional=True, dropout=do if nl > 1 else 0)
        self.attn = nn.Sequential(nn.Linear(h*2, 1), nn.Tanh())
        self.fft_enc = nn.Sequential(nn.Linear(fft_f, 32), nn.ReLU(), nn.Dropout(do), nn.Linear(32, 32), nn.ReLU())
        self.cls = nn.Sequential(nn.Linear(h*2+32, 128), nn.ReLU(), nn.Dropout(do),
                                 nn.Linear(128, 64), nn.ReLU(), nn.Dropout(do*0.5), nn.Linear(64, 2))
    def forward(self, seq, fft):
        o, _ = self.lstm(seq)
        w = torch.softmax(self.attn(o), dim=1)
        ctx = (o * w).sum(dim=1)
        fe = self.fft_enc(fft)
        return self.cls(torch.cat([ctx, fe], dim=1)), w

model = C2BeaconDetector().to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
EPOCHS = 30
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}
best_f1 = 0
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    tloss = 0
    for seq, fft, lab in tqdm(train_loader, desc=f'Epoch {epoch+1}', leave=False):
        seq, fft, lab = seq.to(device), fft.to(device), lab.to(device)
        optimizer.zero_grad()
        logits, _ = model(seq, fft)
        loss = criterion(logits, lab)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tloss += loss.item() * seq.size(0)
    
    model.eval()
    vloss, preds, labs, probs = 0, [], [], []
    with torch.no_grad():
        for seq, fft, lab in test_loader:
            seq, fft, lab = seq.to(device), fft.to(device), lab.to(device)
            logits, _ = model(seq, fft)
            vloss += criterion(logits, lab).item() * seq.size(0)
            preds.extend(logits.argmax(1).cpu().numpy())
            labs.extend(lab.cpu().numpy())
            probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
    
    scheduler.step()
    tl = tloss / len(train_ds)
    vl = vloss / len(test_ds)
    vf = f1_score(labs, preds)
    va = roc_auc_score(labs, probs)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_f1'].append(vf)
    history['val_auc'].append(va)
    
    if vf > best_f1:
        best_f1 = vf
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'c2_best_model.pt'))
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | TL: {tl:.4f} | VL: {vl:.4f} | F1: {vf:.4f} | AUC: {va:.4f}')

total_time = time.time() - start_time
print(f'\nDone in {total_time/60:.1f} min | Best F1: {best_f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_f1'], color='green', linewidth=2)
axes[1].set_title('Validation F1', fontweight='bold'); axes[1].grid(alpha=0.3)
axes[2].plot(history['val_auc'], color='purple', linewidth=2)
axes[2].set_title('Validation AUC-ROC', fontweight='bold'); axes[2].grid(alpha=0.3)
plt.suptitle('C2 Beacon Detector (BiLSTM+FFT)', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'c2_training_curves.png'), dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'c2_best_model.pt')))
model.eval()
preds, labs, probs = [], [], []
with torch.no_grad():
    for seq, fft, lab in test_loader:
        seq, fft = seq.to(device), fft.to(device)
        logits, _ = model(seq, fft)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(lab.numpy())
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())

print('=' * 50)
print('C2 BEACON DETECTOR RESULTS')
print('=' * 50)
print(classification_report(labs, preds, target_names=['Benign', 'C2 Beacon']))
print(f'AUC-ROC: {roc_auc_score(labs, probs):.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(labs, preds)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Purples',
            xticklabels=['Benign', 'C2 Beacon'], yticklabels=['Benign', 'C2 Beacon'], ax=ax)
ax.set_title(f'Confusion Matrix (F1: {f1_score(labs, preds):.4f})', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'c2_confusion_matrix.png'), dpi=150)
plt.show()

## 5. Export to ONNX

In [ ]:
class C2BeaconONNX(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, seq, fft):
        logits, _ = self.m(seq, fft)
        return logits

model_cpu = C2BeaconONNX(model.cpu())
model_cpu.eval()

dummy_seq = torch.randn(1, SEQ_LEN, N_FEATURES)
dummy_fft = torch.randn(1, 5)
onnx_path = os.path.join(OUTPUT_DIR, 'c2_beacon_bilstm.onnx')

torch.onnx.export(
    model_cpu, (dummy_seq, dummy_fft), onnx_path,
    export_params=True, opset_version=14,
    input_names=['flow_sequence', 'fft_features'],
    output_names=['logits'],
    dynamic_axes={'flow_sequence': {0: 'batch'}, 'fft_features': {0: 'batch'}, 'logits': {0: 'batch'}}
)
print(f'ONNX saved: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
r = sess.run(None, {'flow_sequence': np.random.randn(1, SEQ_LEN, N_FEATURES).astype(np.float32),
                     'fft_features': np.random.randn(1, 5).astype(np.float32)})
print(f'ONNX output: {r[0].shape}')

import timeit
ts = np.random.randn(1, SEQ_LEN, N_FEATURES).astype(np.float32)
tf = np.random.randn(1, 5).astype(np.float32)
t = timeit.timeit(lambda: sess.run(None, {'flow_sequence': ts, 'fft_features': tf}), number=1000)
print(f'Inference: {t/1000*1000:.3f} ms/sample')

In [ ]:
metrics = {
    'model_name': 'NetSentinel C2 Beacon Detector',
    'model_type': 'BiLSTM + FFT Periodicity Fusion',
    'version': '1.0.0',
    'f1': float(f1_score(labs, preds)),
    'auc_roc': float(roc_auc_score(labs, probs)),
    'training_minutes': float(total_time / 60),
    'seq_len': SEQ_LEN, 'n_features': N_FEATURES, 'fft_features': 5,
    'data_sources': {
        'beacon': 'CTU-13 botnet', 'benign': 'CIC-IDS2017 + CTU-13 Normal',
        'real_beacon': len(beacon_seqs),
        'real_benign': len(benign_seqs_cic) + len(normal_seqs_ctu),
        'synthetic': need_b + need_n
    }
}
with open(os.path.join(OUTPUT_DIR, 'c2_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print('\n' + '='*50)
print('MODEL CARD: C2 Beacon Detector')
print('='*50)
print(f'  Arch:  BiLSTM + FFT Fusion')
print(f'  Data:  CTU-13 + CIC-IDS2017')
print(f'  F1:    {metrics["f1"]:.4f}')
print(f'  AUC:   {metrics["auc_roc"]:.4f}')
print(f'  ONNX:  {onnx_path}')
print('='*50)

In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/netsentinel_c2_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        fp = os.path.join(OUTPUT_DIR, f)
        zf.write(fp, f)
        print(f'  {f} ({os.path.getsize(fp)/1024:.1f} KB)')

print(f'\nZip: {os.path.getsize(zip_path)/1024/1024:.1f} MB')
FileLink(zip_path)